# Activation Functions

A neural network is built from layers of perceptrons, each computing a weighted sum of its inputs and then passing the result through an **activation function**. Without activation functions, stacking layers would still produce a linear model — it's the activation functions that introduce the non-linearity that lets neural networks approximate complex relationships.

We'll cover five activation functions: **Binary Step**, **ReLU**, **Leaky ReLU**, **TanH**, and **Sigmoid**. Each has a different shape and different practical trade-offs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## Binary Step Function

The binary step function (also called the Heaviside step function) is the activation used in the basic perceptron:

$$f(x) = \begin{cases} 0 & x \leq 0 \\ 1 & x > 0 \end{cases}$$

**When to use it:** When you need a hard binary output — the node either fires or it doesn't. This is fine for a single perceptron classifying linearly separable data, but it's rarely used in deeper networks because its derivative is zero everywhere, which makes gradient-based training impossible.

In [ ]:
x = np.linspace(-6, 6, 500)

def binary_step(x):
    return np.where(x > 0, 1.0, 0.0)

plt.figure(figsize=(6, 3))
plt.plot(x, binary_step(x), color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.ylim(-0.2, 1.4)
plt.title('Binary Step Function')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.tight_layout()
plt.show()

## ReLU (Rectified Linear Unit)

ReLU is the most widely used activation function in modern deep networks:

$$f(x) = \begin{cases} 0 & x \leq 0 \\ x & x > 0 \end{cases}$$

**When to use it:** ReLU works well in hidden layers of deep networks. For negative inputs it returns zero (the node is "off"); for positive inputs it passes the value through unchanged. Its derivative is 1 for positive inputs, which means gradients flow easily during backpropagation — ReLU networks tend to train faster than sigmoid or tanh networks.

**Drawback:** Neurons can get stuck in the "off" state permanently (the *dying ReLU* problem) if weights push all inputs negative and the gradient becomes zero.

In [ ]:
def relu(x):
    return np.maximum(0, x)

plt.figure(figsize=(6, 3))
plt.plot(x, relu(x), color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.title('ReLU')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.tight_layout()
plt.show()

## Leaky ReLU

Leaky ReLU addresses the dying ReLU problem by allowing a small nonzero gradient for negative inputs:

$$f(x) = \begin{cases} ax & x \leq 0 \\ x & x > 0 \end{cases}$$

where $a$ is a small constant, typically 0.01.

**When to use it:** A drop-in replacement for ReLU when dying neurons are a concern. Because the derivative on the negative side is $a$ rather than 0, neurons can recover during training.

In [ ]:
def leaky_relu(x, a=0.1):
    return np.where(x > 0, x, a * x)

plt.figure(figsize=(6, 3))
plt.plot(x, leaky_relu(x), color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.title('Leaky ReLU (a=0.1)')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.tight_layout()
plt.show()

## TanH (Hyperbolic Tangent)

$$f(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

TanH maps any real number to $(-1, 1)$. It is differentiable everywhere, and its output is centered at zero — meaning activations can be negative, which can help gradient flow in some architectures.

**When to use it:** Hidden layers where you want a smooth, bounded, zero-centered activation. TanH was the standard hidden-layer activation before ReLU became dominant. It is still preferred in recurrent networks (LSTMs, GRUs) and in cases where negative activations carry meaning.

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(x, np.tanh(x), color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.ylim(-1.4, 1.4)
plt.title('TanH')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.tight_layout()
plt.show()

## Sigmoid

$$f(x) = \sigma(x) = \frac{1}{1 + e^{-x}}$$

Sigmoid maps any real number to $(0, 1)$. Because the output is always a positive number less than 1, it is natural to interpret it as a probability.

**When to use it:** The **output layer** of a binary classifier — the output is a probability that the example belongs to the positive class. It's less common in hidden layers of deep networks because its gradient becomes very small for large positive or negative inputs (the *vanishing gradient* problem), which slows training.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

plt.figure(figsize=(6, 3))
plt.plot(x, sigmoid(x), color='steelblue', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.ylim(-0.2, 1.2)
plt.title('Sigmoid')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.tight_layout()
plt.show()

## Interactive Comparison

Use the checkboxes below to compare any combination of activation functions on the same axes.

In [ ]:
x_range = np.linspace(-6, 6, 500)

functions = {
    'Binary Step': (binary_step, 'steelblue'),
    'ReLU':        (relu,        'darkorange'),
    'Leaky ReLU':  (lambda x: leaky_relu(x, 0.1), 'seagreen'),
    'TanH':        (np.tanh,    'mediumpurple'),
    'Sigmoid':     (sigmoid,    'crimson'),
}

checkboxes = {name: widgets.Checkbox(value=True, description=name, indent=False)
              for name in functions}

out = widgets.Output()

def update_plot(*args):
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.axvline(0, color='gray', linewidth=0.5)
        for name, (fn, color) in functions.items():
            if checkboxes[name].value:
                ax.plot(x_range, fn(x_range), label=name, color=color, linewidth=2)
        ax.set_ylim(-1.5, 6.5)
        ax.set_xlabel('x')
        ax.set_ylabel('f(x)')
        ax.set_title('Activation Function Comparison')
        ax.legend(loc='upper left')
        plt.tight_layout()
        plt.show()

for cb in checkboxes.values():
    cb.observe(update_plot, names='value')

checkbox_row = widgets.HBox(list(checkboxes.values()))
display(widgets.VBox([checkbox_row, out]))
update_plot()

## Summary: When to Use Each

| Function | Output range | Differentiable? | Typical use |
|---|---|---|---|
| Binary Step | {0, 1} | No | Simple perceptron only |
| ReLU | [0, ∞) | Yes (except x=0) | Hidden layers in deep networks |
| Leaky ReLU | (-∞, ∞) | Yes (except x=0) | Hidden layers when dying ReLU is a concern |
| TanH | (-1, 1) | Yes | Hidden layers, recurrent networks |
| Sigmoid | (0, 1) | Yes | Binary classification output layer |